# Token Caching in a LangGraph ReAct Agent

This tutorial measures prompt caching inside the same two-node ReAct loop used in `01_react_agent.ipynb`:

```text
START -> assistant -> [tools_condition] -> tools -> assistant -> ... -> END
```

**Audience**

- Developers who already know basic LangGraph tool calling and use SAP Generative AI Hub.

**Prerequisites**

- SAP AI Core credentials in the adjacent `.env` file.
- Running deployments for the configured OpenAI, Anthropic, and Gemini models.
- The dependencies from `requirements.txt` installed in the repository `.venv`.

**Learning goals**

- Keep a ReAct transcript cache-friendly.
- Apply each provider's cache strategy at the LLM call boundary.
- Normalize and interpret cache-read, cache-write, uncached-input, and total-input tokens.

> Running all defaults produces about 27 model calls. Results depend on the tenant, model deployment, and SDK version.

## Outline

1. Configure providers and a deterministic multi-step tool.
2. Build one instrumented ReAct graph for all providers.
3. Run OpenAI, Claude on Bedrock, and Gemini.
4. Compare per-call and cumulative cache telemetry.
5. Review best practices and try a cache-busting exercise.

## What caching changes

Caching does **not** make the logical prompt disappear. The provider still reports the complete prompt-side volume as `input_total_tokens`. Caching changes how much of that volume is processed as new input:

| Field | Meaning |
|---|---|
| `input_tokens` | Uncached prompt tokens processed as new input |
| `input_total_tokens` | Prompt-side tokens including cache reads and writes |
| `cache_read_input_tokens` | Prompt tokens reused from provider cache |
| `cache_write_tokens` | Prompt tokens written to the provider cache on this call |

The reliable signal is a sequence of calls where total prompt volume grows while uncached input stays flat or grows much more slowly. This notebook intentionally does not estimate monetary savings because provider prices change independently of token telemetry.

## 1. Imports and environment

The notebook uses SAP Gen AI Hub LangChain wrappers because the models run inside a LangGraph tool loop. Usage is normalized by the adjacent, stdlib-only `normalize_usage.py` helper.

In [1]:
from __future__ import annotations

from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from gen_ai_hub.proxy.langchain.amazon import ChatBedrockConverse
from gen_ai_hub.proxy.langchain.google_genai import ChatGoogleGenerativeAI
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI

from normalize_usage import add_derived_cache_writes, normalize_usage

load_dotenv(".env", override=True)
print("Environment loaded.")

Environment loaded.


## 2. Configuration

The defaults are models validated while this notebook was authored. If one is not deployed in your SAP AI Core resource group, change its model string explicitly; do not silently substitute a different model.

Gemini uses a larger stable prefix because implicit caching may not engage below its provider threshold. `OBSERVATION_FACTS` makes each ordered tool result large enough for transcript growth to be visible.

In [2]:
PROVIDER_CONFIG = {
    "openai": {
        "model": "gpt-5.4",
        "strategy": "passive",
        "stable_facts": 400,
    },
    "bedrock": {
        "model": "anthropic--claude-4.6-sonnet",
        "strategy": "explicit cache_control",
        "stable_facts": 400,
    },
    "gemini": {
        "model": "gemini-3.5-flash",
        "strategy": "passive implicit",
        "stable_facts": 1200,
    },
}

ENABLED_PROVIDERS = ["openai", "bedrock", "gemini"]
DEMO_STEPS = 8
OBSERVATION_FACTS = 60
MAX_OUTPUT_TOKENS = 400

pd.DataFrame.from_dict(PROVIDER_CONFIG, orient="index")

,model,strategy,stable_facts
openai,gpt-5.4,passive,400
bedrock,anthropic--claude-4.6-sonnet,explicit cache_control,400
gemini,gemini-3.5-flash,passive implicit,1200


## 3. Deterministic ordered tool

The agent must request eight steps in order and wait for every observation before continuing. This creates nine assistant/model calls per provider: one before each tool step and one final answer.

The generated reference text is deliberately synthetic. In a real agent, the stable prefix would be a system policy, tool schemas, or other unchanged context—not filler added merely to trigger caching.

In [3]:
def build_stable_reference(fact_count: int) -> str:
    """Return deterministic reference text large enough for cache probing."""

    if fact_count < 1:
        raise ValueError("fact_count must be positive")
    return " ".join(
        f"fact-{index}: this reference value is stable."
        for index in range(1, fact_count + 1)
    )


@tool
def read_demo_step(step: int) -> str:
    """Return one ordered observation for steps 1 through DEMO_STEPS."""

    if not 1 <= step <= DEMO_STEPS:
        raise ValueError(f"step must be between 1 and {DEMO_STEPS}")
    evidence = " ".join(
        f"observation-{step}-{index}: deterministic tool evidence."
        for index in range(1, OBSERVATION_FACTS + 1)
    )
    next_action = (
        f"Next, call read_demo_step with step {step + 1}."
        if step < DEMO_STEPS
        else "All steps are complete. Return one short final sentence."
    )
    return f"Step {step} completed. {evidence} {next_action}"


TOOLS = [read_demo_step]
print(f"Configured {DEMO_STEPS} ordered tool steps.")

Configured 8 ordered tool steps.


## 4. Provider adapters

Only the model adapter and cache policy vary:

- **OpenAI:** passive prompt caching; no cache flag.
- **Claude on Bedrock:** explicit `cache_control`, applied after binding tools.
- **Gemini:** passive implicit caching; explicit cached contents may not route through SAP Gen AI Hub.

The system prompt is always a `SystemMessage`, and `MessagesState` keeps all user, assistant, and tool messages append-only.

In [4]:
def make_tool_bound_model(provider: str, model_name: str):
    """Create a SAP Gen AI Hub model with the demo tool and cache policy."""

    if provider == "openai":
        model = ChatOpenAI(
            proxy_model_name=model_name,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
    elif provider == "bedrock":
        model = ChatBedrockConverse(
            model_name=model_name,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
    elif provider == "gemini":
        model = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0.0,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        )
    else:
        raise ValueError(f"Unsupported provider: {provider}")

    bound_model = model.bind_tools(TOOLS)
    if provider == "bedrock":
        bound_model = bound_model.bind(cache_control={"type": "default"})
    return bound_model

## 5. Instrument the assistant node

Caching belongs at the real provider call, inside the `assistant` node. Tool nodes do not call the model and therefore do not emit usage rows.

A fresh run ID salts the stable system prompt once per provider. This prevents an old notebook run from warming call 1 while keeping the prefix byte-identical inside the active ReAct loop.

In [5]:
def build_system_prompt(run_id: str, stable_facts: int) -> str:
    """Return one stable system prompt for an entire provider run."""

    instructions = f"""You are running a controlled ReAct cache demonstration.
Run ID: {run_id}
Call read_demo_step for steps 1 through {DEMO_STEPS}, strictly in order.
Make exactly one tool call per assistant response and wait for its ToolMessage.
Never skip, repeat, combine, or parallelize steps.
After step {DEMO_STEPS}, return one short sentence confirming completion.

Stable reference material follows:
"""
    return instructions + build_stable_reference(stable_facts)


def build_instrumented_agent(
    provider: str,
    model_name: str,
    stable_facts: int,
    *,
    bust_cache: bool = False,
):
    """Compile a ReAct agent and return it with its usage-row buffer."""

    model = make_tool_bound_model(provider, model_name)
    usage_rows: list[dict] = []
    stable_prompt = build_system_prompt(str(uuid4()), stable_facts)

    def assistant(state: MessagesState) -> dict:
        """Call the model once, record normalized usage, and append its reply."""

        system_prompt = stable_prompt
        if bust_cache:
            system_prompt += f"\nCache-busting call nonce: {uuid4()}"
        response = model.invoke([SystemMessage(content=system_prompt), *state["messages"]])
        usage = normalize_usage(provider, response)
        usage.update(
            {
                "provider": provider,
                "model": model_name,
                "call": len(usage_rows) + 1,
            }
        )
        usage_rows.append(usage)
        return {"messages": [response]}

    graph = StateGraph(MessagesState)
    graph.add_node("assistant", assistant)
    graph.add_node("tools", ToolNode(tools=TOOLS, handle_tool_errors=True))
    graph.add_edge(START, "assistant")
    graph.add_conditional_edges("assistant", tools_condition)
    graph.add_edge("tools", "assistant")
    return graph.compile(), usage_rows

## 6. Run the three providers

Each provider receives a separate cold-start nonce and the same ordered task. The cell stops on deployment or runtime errors rather than choosing another model automatically.

In [6]:
def run_provider(provider: str, *, bust_cache: bool = False) -> tuple[dict, list[dict]]:
    """Run one provider and validate the expected sequential ReAct trace."""

    config = PROVIDER_CONFIG[provider]
    agent, usage_rows = build_instrumented_agent(
        provider,
        config["model"],
        config["stable_facts"],
        bust_cache=bust_cache,
    )
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Execute all {DEMO_STEPS} cache-demonstration steps now."
                )
            ]
        },
        {"recursion_limit": 30},
    )
    add_derived_cache_writes(usage_rows)

    tool_messages = [
        message for message in result["messages"] if isinstance(message, ToolMessage)
    ]
    expected_calls = DEMO_STEPS + 1
    if len(tool_messages) != DEMO_STEPS or len(usage_rows) != expected_calls:
        raise RuntimeError(
            f"{provider} did not follow the ordered probe: "
            f"{len(tool_messages)} tool results and {len(usage_rows)} model calls."
        )
    return result, usage_rows


RUN_RESULTS: dict[str, dict] = {}
USAGE_BY_PROVIDER: dict[str, list[dict]] = {}

for provider in ENABLED_PROVIDERS:
    print(f"Running {provider} with {PROVIDER_CONFIG[provider]['model']}...")
    result, rows = run_provider(provider)
    RUN_RESULTS[provider] = result
    USAGE_BY_PROVIDER[provider] = rows
    print(f"  completed with {len(rows)} model calls")

Running openai with gpt-5.4...
  completed with 9 model calls
Running bedrock with anthropic--claude-4.6-sonnet...
  completed with 9 model calls
Running gemini with gemini-3.5-flash...
  completed with 9 model calls


## 7. Per-call telemetry

The table deliberately omits raw provider metadata and keeps only comparable fields. A zero on call 1 is expected for a cold run. Cache behavior should be judged across the complete sequence.

In [7]:
DISPLAY_COLUMNS = [
    "provider",
    "call",
    "input_tokens",
    "input_total_tokens",
    "cache_read_input_tokens",
    "cache_write_tokens",
    "output_tokens",
    "total_tokens",
    "cache_hit_rate_pct",
]


def usage_frame(rows: list[dict]) -> pd.DataFrame:
    """Return a display-ready per-call usage table.

    A single ``cache_write_tokens`` column reports the tokens written to the
    provider cache on each call. Providers that expose writes directly are used
    as-is; the remaining providers fall back to the estimate produced by
    ``add_derived_cache_writes``.
    """

    frame = pd.DataFrame(rows).copy()
    prompt_total = frame["input_total_tokens"].fillna(0)
    cache_read = frame["cache_read_input_tokens"].fillna(0)
    frame["cache_hit_rate_pct"] = (
        cache_read.div(prompt_total.where(prompt_total > 0, 1)).mul(100).round(1)
    )
    reported_write = frame["cache_write_input_tokens"]
    fallback_write = frame.get("cache_write_derived_tokens")
    frame["cache_write_tokens"] = reported_write.fillna(fallback_write)
    return frame[DISPLAY_COLUMNS]


all_usage = pd.concat(
    [usage_frame(rows) for rows in USAGE_BY_PROVIDER.values()],
    ignore_index=True,
)
display(all_usage.fillna("—"))

,provider,call,input_tokens,input_total_tokens,cache_read_input_tokens,cache_write_tokens,output_tokens,total_tokens,cache_hit_rate_pct
0,openai,1,4244,4244,0,0.0,19,4263,0.0
1,openai,2,4890,4890,0,4864.0,19,4909,0.0
2,openai,3,672,5536,4864,0.0,19,5555,87.9
3,openai,4,1958,6182,4224,1920.0,19,6201,68.3
4,openai,5,684,6828,6144,640.0,19,6847,90.0
5,openai,6,690,7474,6784,640.0,19,7493,90.8
6,openai,7,696,8120,7424,640.0,19,8139,91.4
7,openai,8,702,8766,8064,640.0,19,8785,92.0
8,openai,9,708,9412,8704,—,15,9427,92.5
9,bedrock,1,3,4693,0,4690,79,4772,0.0


## 8. Cumulative comparison

The cumulative cache-hit rate is `cache_read_input_tokens / input_total_tokens`. It is a telemetry ratio, not a price discount. Provider pricing may assign different rates to reads, writes, and uncached tokens.

In [8]:
def summarize_usage(provider: str, rows: list[dict]) -> dict:
    """Return cumulative cache metrics for one provider run."""

    frame = pd.DataFrame(rows)
    input_total = int(frame["input_total_tokens"].fillna(0).sum())
    cache_read = int(frame["cache_read_input_tokens"].fillna(0).sum())
    reported_write = frame["cache_write_input_tokens"]
    fallback_write = frame.get("cache_write_derived_tokens")
    cache_write = int(reported_write.fillna(fallback_write).fillna(0).sum())
    return {
        "provider": provider,
        "model": PROVIDER_CONFIG[provider]["model"],
        "strategy": PROVIDER_CONFIG[provider]["strategy"],
        "model_calls": len(frame),
        "uncached_input_tokens": int(frame["input_tokens"].fillna(0).sum()),
        "input_total_tokens": input_total,
        "cache_read_input_tokens": cache_read,
        "cache_write_tokens": cache_write,
        "output_tokens": int(frame["output_tokens"].fillna(0).sum()),
        "cache_hit_rate_pct": round(100 * cache_read / input_total, 1)
        if input_total
        else 0.0,
    }


summary = pd.DataFrame(
    [
        summarize_usage(provider, USAGE_BY_PROVIDER[provider])
        for provider in ENABLED_PROVIDERS
    ]
)
display(summary)

,provider,model,strategy,model_calls,uncached_input_tokens,input_total_tokens,cache_read_input_tokens,cache_write_tokens,output_tokens,cache_hit_rate_pct
0,openai,gpt-5.4,passive,9,15244,61452,46208,9344,167,75.2
1,bedrock,anthropic--claude-4.6-sonnet,explicit cache_control,9,11,69613,58835,10767,587,84.5
2,gemini,gemini-3.5-flash,passive implicit,9,28006,157968,129962,18296,395,82.3


## How to interpret the result

- Look for nonzero cache reads on several consecutive calls—not a single isolated hit.
- Compare `input_tokens` with `input_total_tokens` as the transcript grows. Successful caching keeps the uncached share much smaller.
- Bedrock keeps uncached `input_tokens` near zero because `cache_control` cache-points the system prompt, tools, and the latest message, so each new tool observation is written to the cache and read back on the following call.
- The `cache_write_tokens` column reports how much was written to the cache on each call. OpenAI and Gemini do not always expose writes, so those rows fall back to an estimate.
- A cache read on call 1 usually means an earlier run warmed an identical prefix. The per-run nonce is intended to prevent that.

Raw token totals should **not** be compared as a model-quality or provider-cost benchmark: the providers tokenize differently and use different prefix thresholds.

## When aggressive caching can backfire

Cache-pointing the latest message on every turn is a good fit for this append-only ReAct loop, but it is not free and not universally correct. Caching is a bet: you pay to write tokens now, expecting to read them back cheaply on later calls. Understand the tradeoffs before copying this pattern into a different workload.

- **Writes carry a premium.** Writing tokens to the cache costs modestly more than processing them as ordinary input. That premium only pays off if the written prefix is actually read back on a later call. In a tight tool loop each observation is reused on the next turn, so the bet wins; in a short one- or two-turn exchange the write may never be read and becomes pure overhead.
- **Cache entries expire.** Provider caches have a short time-to-live (a few minutes). If a slow step sits between turns—a long-running tool, an external API, or a human-in-the-loop pause—the cache can expire and the next call pays the full write again.
- **The prefix is fragile.** Any byte that changes before a cache point invalidates everything after it. Timestamps, nonces, reordered tool definitions, or edited earlier turns all trigger a full re-write. The cache-busting exercise below shows this directly.
- **Small prompts sit below the threshold.** Providers only cache once the stable prefix crosses a minimum size. Below that threshold writes may still be billed while no reads ever trigger—cost without benefit. Increase the stable prefix before concluding a model cannot cache.
- **Match the strategy to the workload.** Aggressive per-turn cache points suit long, linear, append-only agent loops with fast turns and a large reused prefix. Branching conversations, short sessions, or rapidly changing context may cache less effectively or lose money.

## Best practices

1. Keep the active transcript append-only; do not flatten, reorder, summarize, or rewrite earlier turns during the cache test.
2. Put stable instructions in `SystemMessage` and do not duplicate them into user content.
3. Keep stable context and tool schemas ahead of frequently changing user/tool data.
4. Use passive caching for OpenAI and Gemini unless telemetry proves another supported path.
5. Use explicit Bedrock cache points through `cache_control` for Claude.
6. Test multiple consecutive calls and use a cold-run nonce before making claims.
7. Increase stable prefix size before concluding that Gemini or a high-threshold model cannot cache.
8. Record token usage at the exact `llm.invoke`/`llm.ainvoke` boundary.

Do not trim or summarize messages inside this demonstration: doing so changes the prefix being measured. Production agents may need context management, but that should be evaluated as a separate tradeoff.

## Exercise: deliberately break the prefix

Set `RUN_CACHE_BUSTING_EXERCISE = True` and rerun the cell below. It adds a different nonce to the system prompt on every model call. Predict what happens to cache reads before executing it.

The exercise is disabled by default because it repeats paid model calls.

In [9]:
RUN_CACHE_BUSTING_EXERCISE = False
EXERCISE_PROVIDER = "openai"

if RUN_CACHE_BUSTING_EXERCISE:
    _, exercise_rows = run_provider(EXERCISE_PROVIDER, bust_cache=True)
    display(usage_frame(exercise_rows).fillna("—"))
else:
    print("Exercise skipped. Set RUN_CACHE_BUSTING_EXERCISE = True to run it.")

Exercise skipped. Set RUN_CACHE_BUSTING_EXERCISE = True to run it.


### Exercise answer

Changing the system prefix on every call prevents the provider from matching the earlier prefix. Cache reads should fall to zero or become materially smaller, while uncached input approaches total prompt-side input. If reads remain high, check whether another stable prefix—such as tool schemas—still exceeds the provider threshold.

## Troubleshooting

- **Deployment not found:** edit the model string in `PROVIDER_CONFIG`; do not guess another deployment.
- **Zero cache reads:** increase `stable_facts`, keep the prompt byte-identical, and inspect several turns. Gemini commonly needs the largest prefix.
- **Bedrock has zero reads:** confirm the model is `ChatBedrockConverse` and `cache_control` is bound after tools.
- **Unexpected number of calls:** strengthen the ordered-tool instruction or inspect tool-call messages for parallel/duplicate requests.
- **Call 1 is already cached:** restart with a fresh run ID; do not reuse an old nonce when measuring cold start.
- **Usage fields are missing:** inspect `response.usage_metadata` and `response.response_metadata`, then update `normalize_usage.py` for the installed SDK shape.

## Summary

One simple ReAct graph can demonstrate three different provider strategies. The architecture stays unchanged; only the model adapter, Bedrock cache control, and normalized telemetry differ.

The key success signal is not lower logical prompt volume. It is growing `input_total_tokens` accompanied by much slower growth in uncached `input_tokens`, backed by consecutive cache-read observations.